# 醫療品質與裝置檢驗

> **課程定位**：實務應用篇（5/5）  
> **前置課程**：[04_產後憂鬱研究案例](./04_產後憂鬱研究案例.ipynb)  
> **學習路徑**：01 基礎框架 → 02 臨床試驗 → 03 流行病學 → 04 產後憂鬱案例 → **05 醫療品質**

## 學習目標
- 應用 one-sample t 檢定於醫療器材品質檢驗
- 使用 Kruskal-Wallis 分析病患滿意度問卷
- 比較醫院間再入院率（比例檢定與漏斗圖）
- 評估診斷工具準確度（McNemar's test, ROC/AUC）
- 了解評估者間信度（Cohen's Kappa）

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'Microsoft JhengHei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

## 1. 醫療器材品質檢驗

### 情境：胰島素注射筆的劑量準確度

某醫療器材公司生產胰島素注射筆，規格要求每次注射量為 **10.0 ± 0.3 IU**（國際單位）。
品管部門抽檢 50 支注射筆，檢驗注射量是否符合規格。

**統計問題**：
1. 平均注射量是否等於標定值 10.0 IU？（one-sample t-test）
2. 製程能力是否足夠？（Cp/Cpk）

In [ ]:
np.random.seed(42)
n_devices = 50

# 模擬注射筆數據（略微偏高）
injection_volume = np.random.normal(loc=10.05, scale=0.08, size=n_devices)

target = 10.0
USL = 10.3
LSL = 9.7

# One-sample t-test
t_stat, p_value = stats.ttest_1samp(injection_volume, target)

# 描述統計
mean_vol = injection_volume.mean()
std_vol = injection_volume.std(ddof=1)

# 95% CI
ci = stats.t.interval(0.95, df=n_devices-1, loc=mean_vol, scale=std_vol/np.sqrt(n_devices))

# Cp, Cpk
Cp = (USL - LSL) / (6 * std_vol)
Cpu = (USL - mean_vol) / (3 * std_vol)
Cpl = (mean_vol - LSL) / (3 * std_vol)
Cpk = min(Cpu, Cpl)

print("=" * 60)
print("胰島素注射筆品質檢驗")
print("=" * 60)
print(f"\n  抽檢數量: {n_devices} 支")
print(f"  平均注射量: {mean_vol:.4f} IU")
print(f"  標準差: {std_vol:.4f} IU")
print(f"  95% CI: ({ci[0]:.4f}, {ci[1]:.4f})")
print(f"\n  One-sample t-test (H\u2080: \u03bc = {target}):")
print(f"  t = {t_stat:.3f}, p = {p_value:.4f}")
print(f"  結論: {'平均值與標定值有顯著差異 \u26a0\ufe0f' if p_value < 0.05 else '平均值與標定值無顯著差異 \u2705'}")
print(f"\n  製程能力:")
print(f"  Cp  = {Cp:.3f}")
print(f"  Cpk = {Cpk:.3f} ({'合格 \u2705' if Cpk >= 1.33 else '需改善 \u26a0\ufe0f'})")

# 視覺化
fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(injection_volume, bins=15, density=True, alpha=0.7, color='steelblue', edgecolor='white')
x_fit = np.linspace(injection_volume.min()-0.1, injection_volume.max()+0.1, 200)
ax.plot(x_fit, stats.norm.pdf(x_fit, mean_vol, std_vol), 'b-', linewidth=2)
ax.axvline(x=target, color='green', linewidth=2, linestyle='-', label=f'標定值 = {target}')
ax.axvline(x=USL, color='red', linewidth=2, linestyle='--', label=f'USL = {USL}')
ax.axvline(x=LSL, color='red', linewidth=2, linestyle='--', label=f'LSL = {LSL}')
ax.axvline(x=mean_vol, color='orange', linewidth=2, label=f'Mean = {mean_vol:.3f}')
ax.set_title(f'胰島素注射筆劑量分布 (Cpk = {Cpk:.2f})', fontsize=14, fontweight='bold')
ax.set_xlabel('注射量 (IU)')
ax.set_ylabel('機率密度')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 2. 病患滿意度調查分析

### 情境：三個院區的急診病患滿意度

某醫療體系包含 A、B、C 三個院區，使用 Likert 5 點量表（1=非常不滿意 ~ 5=非常滿意）調查急診病患滿意度。

**統計問題**：三個院區的滿意度是否有顯著差異？

由於 Likert 量表為**次序資料**，適合使用非參數的 **Kruskal-Wallis 檢定**。

In [ ]:
np.random.seed(2024)

# 模擬滿意度數據
satisfaction = {
    'A院區': np.random.choice([1,2,3,4,5], 80, p=[0.05, 0.10, 0.25, 0.40, 0.20]),
    'B院區': np.random.choice([1,2,3,4,5], 90, p=[0.10, 0.15, 0.30, 0.30, 0.15]),
    'C院區': np.random.choice([1,2,3,4,5], 70, p=[0.03, 0.07, 0.20, 0.35, 0.35]),
}

# Kruskal-Wallis
H_stat, p_kw = stats.kruskal(*satisfaction.values())

print("=" * 60)
print("病患滿意度分析 (Kruskal-Wallis)")
print("=" * 60)

for name, data in satisfaction.items():
    print(f"  {name} (n={len(data)}): 中位數 = {np.median(data):.0f}, 平均 = {data.mean():.2f}")

print(f"\nKruskal-Wallis: H = {H_stat:.3f}, p = {p_kw:.4f}")

# Post-hoc: 兩兩 Mann-Whitney U (Bonferroni)
from itertools import combinations
print(f"\nPost-hoc 兩兩比較 (Bonferroni \u03b1/3 = {0.05/3:.4f}):")
for g1, g2 in combinations(satisfaction.keys(), 2):
    stat_mw, p_mw = stats.mannwhitneyu(satisfaction[g1], satisfaction[g2])
    sig = '***' if p_mw < 0.05/3 else 'ns'
    print(f"  {g1} vs {g2}: U = {stat_mw:.0f}, p = {p_mw:.4f} {sig}")

# 視覺化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 堆疊長條圖
labels = [1, 2, 3, 4, 5]
x_pos = np.arange(len(satisfaction))
width = 0.6
bottoms = {name: np.zeros(5) for name in satisfaction}

colors_likert = ['#d32f2f', '#ff9800', '#ffc107', '#8bc34a', '#4caf50']
label_texts = ['非常不滿意', '不滿意', '普通', '滿意', '非常滿意']

for idx, (level, color, text) in enumerate(zip(labels, colors_likert, label_texts)):
    proportions = [np.mean(satisfaction[g] == level) * 100 for g in satisfaction.keys()]
    bottom_vals = [sum(np.mean(satisfaction[g] == l) * 100 for l in range(1, level)) for g in satisfaction.keys()]
    axes[0].bar(x_pos, proportions, width, bottom=bottom_vals, color=color, label=text, edgecolor='white')

axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(satisfaction.keys(), fontsize=12)
axes[0].set_ylabel('比例 (%)')
axes[0].set_title('滿意度分布', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=8, loc='upper right')

# 箱形圖
bp = axes[1].boxplot(satisfaction.values(), labels=satisfaction.keys(), patch_artist=True,
                      medianprops=dict(color='red', linewidth=2))
for patch, color in zip(bp['boxes'], ['coral', 'steelblue', 'mediumseagreen']):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_ylabel('滿意度分數')
axes[1].set_title(f'各院區滿意度比較 (p = {p_kw:.4f})', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

## 3. 醫院再入院率比較

### 情境：10 家醫院的 30 天再入院率

衛生主管機關比較 10 家醫院的心衰竭病患 30 天內再入院率。

**統計方法**：
- 比例 z 檢定（兩兩比較）
- **漏斗圖（Funnel Plot）**：將各醫院的再入院率與全國平均比較

In [ ]:
np.random.seed(42)

hospitals = [f'醫院{chr(65+i)}' for i in range(10)]
n_patients = np.random.randint(100, 500, 10)
# 大部分醫院在正常範圍，1-2家偏高
true_rates = np.array([0.15, 0.14, 0.16, 0.22, 0.13, 0.15, 0.14, 0.25, 0.16, 0.15])
readmissions = np.array([np.random.binomial(n, p) for n, p in zip(n_patients, true_rates)])
observed_rates = readmissions / n_patients

# 全國平均
overall_rate = readmissions.sum() / n_patients.sum()

print("=" * 60)
print("30 天再入院率比較")
print("=" * 60)

df_hosp = pd.DataFrame({
    '醫院': hospitals,
    '出院人數': n_patients,
    '再入院數': readmissions,
    '再入院率': observed_rates
})
display(df_hosp.style.format({'再入院率': '{:.1%}'}))
print(f"\n全國平均再入院率: {overall_rate:.1%}")

# 漏斗圖
fig, ax = plt.subplots(figsize=(12, 7))

# 計算管制界限
n_range = np.arange(80, 520)
se = np.sqrt(overall_rate * (1 - overall_rate) / n_range)
ucl_95 = overall_rate + 1.96 * se
lcl_95 = overall_rate - 1.96 * se
ucl_99 = overall_rate + 2.576 * se
lcl_99 = overall_rate - 2.576 * se

ax.fill_between(n_range, lcl_99, ucl_99, alpha=0.1, color='red', label='99% 管制區間')
ax.fill_between(n_range, lcl_95, ucl_95, alpha=0.15, color='orange', label='95% 管制區間')
ax.plot(n_range, ucl_95, 'orange', linewidth=1)
ax.plot(n_range, lcl_95, 'orange', linewidth=1)
ax.plot(n_range, ucl_99, 'red', linewidth=1, linestyle='--')
ax.plot(n_range, lcl_99, 'red', linewidth=1, linestyle='--')
ax.axhline(y=overall_rate, color='green', linewidth=2, label=f'全國平均 = {overall_rate:.1%}')

for i, (n, rate, hosp) in enumerate(zip(n_patients, observed_rates, hospitals)):
    is_outlier = rate > overall_rate + 1.96 * np.sqrt(overall_rate*(1-overall_rate)/n)
    color = 'red' if is_outlier else 'steelblue'
    marker = 's' if is_outlier else 'o'
    ax.scatter(n, rate, color=color, s=80, zorder=5, marker=marker)
    ax.annotate(hosp, (n, rate), textcoords="offset points", xytext=(5, 5), fontsize=9)

ax.set_xlabel('出院人數', fontsize=12)
ax.set_ylabel('30 天再入院率', fontsize=12)
ax.set_title('再入院率漏斗圖（Funnel Plot）', fontsize=16, fontweight='bold')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 標記異常醫院
print("\n異常偵測結果（超出 95% 管制區間）：")
for i in range(len(hospitals)):
    se_i = np.sqrt(overall_rate * (1-overall_rate) / n_patients[i])
    if observed_rates[i] > overall_rate + 1.96 * se_i:
        print(f"  \u26a0\ufe0f {hospitals[i]}: 再入院率 {observed_rates[i]:.1%} (超出 UCL)")

## 4. 診斷工具準確度評估

### 情境：兩種快篩試劑的比較

比較新型 COVID-19 快篩試劑（Test B）與現行試劑（Test A）的診斷表現。
同一批 500 個檢體同時進行兩種快篩和 PCR（金標準）。

### 指標說明

| 指標 | 公式 | 意義 |
|------|------|------|
| 敏感度 (Sensitivity) | TP/(TP+FN) | 真正患病者被正確偵測的比例 |
| 特異度 (Specificity) | TN/(TN+FP) | 真正健康者被正確排除的比例 |
| 陽性預測值 (PPV) | TP/(TP+FP) | 陽性結果中真正患病的比例 |
| 陰性預測值 (NPV) | TN/(TN+FN) | 陰性結果中真正健康的比例 |

In [ ]:
np.random.seed(42)
n_samples = 500
prevalence = 0.15

# 真實疾病狀態
disease = np.random.binomial(1, prevalence, n_samples)

# Test A（敏感度 85%, 特異度 90%）
test_a = np.zeros(n_samples, dtype=int)
for i in range(n_samples):
    if disease[i] == 1:
        test_a[i] = np.random.binomial(1, 0.85)
    else:
        test_a[i] = np.random.binomial(1, 0.10)

# Test B（敏感度 92%, 特異度 88%）
test_b = np.zeros(n_samples, dtype=int)
for i in range(n_samples):
    if disease[i] == 1:
        test_b[i] = np.random.binomial(1, 0.92)
    else:
        test_b[i] = np.random.binomial(1, 0.12)

def diagnostic_metrics(y_true, y_pred, test_name=""):
    """計算診斷指標"""
    TP = np.sum((y_true == 1) & (y_pred == 1))
    TN = np.sum((y_true == 0) & (y_pred == 0))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))
    
    sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0
    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0
    ppv = TP / (TP + FP) if (TP + FP) > 0 else 0
    npv = TN / (TN + FN) if (TN + FN) > 0 else 0
    accuracy = (TP + TN) / (TP + TN + FP + FN)
    
    print(f"\n  {test_name}:")
    print(f"    TP={TP}, FP={FP}, FN={FN}, TN={TN}")
    print(f"    敏感度: {sensitivity:.1%}")
    print(f"    特異度: {specificity:.1%}")
    print(f"    PPV:    {ppv:.1%}")
    print(f"    NPV:    {npv:.1%}")
    print(f"    準確度: {accuracy:.1%}")
    
    return {'TP':TP, 'TN':TN, 'FP':FP, 'FN':FN, 'sens':sensitivity, 'spec':specificity}

print("=" * 60)
print("診斷工具比較")
print("=" * 60)
print(f"樣本數: {n_samples}, 盛行率: {disease.mean():.1%}")

metrics_a = diagnostic_metrics(disease, test_a, "Test A (現行試劑)")
metrics_b = diagnostic_metrics(disease, test_b, "Test B (新型試劑)")

# McNemar's test: 兩種試劑在同一批檢體的表現差異
# 比較不一致的配對
b_only_correct = np.sum((test_b == disease) & (test_a != disease))
a_only_correct = np.sum((test_a == disease) & (test_b != disease))

# McNemar statistic
if b_only_correct + a_only_correct > 0:
    mcnemar_stat = (abs(b_only_correct - a_only_correct) - 1)**2 / (b_only_correct + a_only_correct)
    p_mcnemar = 1 - stats.chi2.cdf(mcnemar_stat, df=1)
else:
    mcnemar_stat, p_mcnemar = 0, 1

print(f"\nMcNemar's test（兩試劑正確率差異）:")
print(f"  Test B 獨自正確: {b_only_correct}")
print(f"  Test A 獨自正確: {a_only_correct}")
print(f"  McNemar \u03c7\u00b2 = {mcnemar_stat:.3f}, p = {p_mcnemar:.4f}")

In [ ]:
# 模擬連續分數（用於 ROC 曲線）
np.random.seed(42)
scores_a = np.where(disease==1, np.random.normal(0.7, 0.2, n_samples), np.random.normal(0.3, 0.2, n_samples))
scores_b = np.where(disease==1, np.random.normal(0.75, 0.18, n_samples), np.random.normal(0.28, 0.2, n_samples))
scores_a = np.clip(scores_a, 0, 1)
scores_b = np.clip(scores_b, 0, 1)

def compute_roc(y_true, scores):
    """手動計算 ROC 曲線"""
    thresholds = np.sort(np.unique(scores))[::-1]
    tpr_list, fpr_list = [0], [0]
    
    for thresh in thresholds:
        y_pred = (scores >= thresh).astype(int)
        TP = np.sum((y_true==1) & (y_pred==1))
        FP = np.sum((y_true==0) & (y_pred==1))
        FN = np.sum((y_true==1) & (y_pred==0))
        TN = np.sum((y_true==0) & (y_pred==0))
        tpr_list.append(TP / (TP+FN) if (TP+FN) > 0 else 0)
        fpr_list.append(FP / (FP+TN) if (FP+TN) > 0 else 0)
    
    tpr_list.append(1)
    fpr_list.append(1)
    auc = np.trapz(sorted(tpr_list), sorted(fpr_list))
    return np.array(fpr_list), np.array(tpr_list), auc

fpr_a, tpr_a, auc_a = compute_roc(disease, scores_a)
fpr_b, tpr_b, auc_b = compute_roc(disease, scores_b)

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(fpr_a, tpr_a, 'b-', linewidth=2, label=f'Test A (AUC = {auc_a:.3f})')
ax.plot(fpr_b, tpr_b, 'r-', linewidth=2, label=f'Test B (AUC = {auc_b:.3f})')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='隨機 (AUC = 0.500)')
ax.set_xlabel('1 - 特異度 (FPR)', fontsize=12)
ax.set_ylabel('敏感度 (TPR)', fontsize=12)
ax.set_title('ROC 曲線比較', fontsize=16, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

print(f"AUC 比較:")
print(f"  Test A: {auc_a:.3f}")
print(f"  Test B: {auc_b:.3f}")
print(f"  差異: {auc_b - auc_a:+.3f}")

## 5. 評估者間信度：Cohen's Kappa

### 情境：臨床評分一致性

兩位醫師對 100 份 X 光片進行判讀（正常/異常），衡量判讀的一致性。

$$\kappa = \frac{P_o - P_e}{1 - P_e}$$

| \u03ba 值 | 一致性等級 |
|------|-----------|
| < 0.20 | 幾乎無一致性 |
| 0.21-0.40 | 輕度一致 |
| 0.41-0.60 | 中度一致 |
| 0.61-0.80 | 高度一致 |
| 0.81-1.00 | 幾乎完全一致 |

In [ ]:
np.random.seed(42)
n_films = 100

# 兩位醫師的判讀（0=正常, 1=異常）
# 設計為高度一致但有部分歧異
true_status = np.random.binomial(1, 0.35, n_films)

doc_a = true_status.copy()
doc_b = true_status.copy()

# 醫師 A 有 10% 誤判率
flip_a = np.random.choice(n_films, int(n_films * 0.10), replace=False)
doc_a[flip_a] = 1 - doc_a[flip_a]

# 醫師 B 有 12% 誤判率
flip_b = np.random.choice(n_films, int(n_films * 0.12), replace=False)
doc_b[flip_b] = 1 - doc_b[flip_b]

# 列聯表
ct = pd.crosstab(pd.Series(doc_a, name='醫師A'), pd.Series(doc_b, name='醫師B'))
ct.index = ['正常', '異常']
ct.columns = ['正常', '異常']

# Cohen's Kappa
Po = (np.sum(doc_a == doc_b)) / n_films
Pe = (np.sum(doc_a==0)*np.sum(doc_b==0) + np.sum(doc_a==1)*np.sum(doc_b==1)) / n_films**2
kappa = (Po - Pe) / (1 - Pe) if (1 - Pe) != 0 else 0

# SE and CI for kappa
se_kappa = np.sqrt(Po * (1-Po) / (n_films * (1-Pe)**2))
kappa_ci = (kappa - 1.96*se_kappa, kappa + 1.96*se_kappa)

print("=" * 60)
print("評估者間信度 (Cohen's Kappa)")
print("=" * 60)
print(f"\n判讀列聯表：")
display(ct)
print(f"\n  觀察一致率 Po = {Po:.3f}")
print(f"  期望一致率 Pe = {Pe:.3f}")
print(f"  Cohen's \u03ba = {kappa:.3f}, 95% CI: ({kappa_ci[0]:.3f}, {kappa_ci[1]:.3f})")

if kappa >= 0.81:
    level = "幾乎完全一致 \u2705"
elif kappa >= 0.61:
    level = "高度一致 \u2705"
elif kappa >= 0.41:
    level = "中度一致 \u26a0\ufe0f"
else:
    level = "一致性不足 \u274c"
print(f"  一致性等級: {level}")

## 6. NBME 臨床筆記評分延伸思考

> 參考資料：[NBME - Score Clinical Patient Notes (Kaggle)](https://www.kaggle.com/code/odins0n/nbme-detailed-eda)

在 NBME（National Board of Medical Examiners）的臨床筆記評分中，假設檢定可應用於：

1. **評分者信度**：多位評分者對同一筆記的評分是否一致（ICC, Fleiss' Kappa）
2. **評分公平性**：不同學生群體（性別、種族）的平均分數是否有差異（t-test, ANOVA）
3. **自動評分系統驗證**：NLP 模型分數與人工評分是否有統計差異（配對 t-test, Bland-Altman plot）
4. **評分標準比較**：不同評分標準產生的分數分布是否相同（Kolmogorov-Smirnov test）

這些方法連結了我們在本系列中學到的所有檢定技巧。

## 7. 本章小結

| 應用場景 | 統計方法 | Python 工具 |
|----------|----------|-------------|
| 器材品質（連續, 單組） | one-sample t-test, Cp/Cpk | `scipy.stats.ttest_1samp` |
| 滿意度（次序, 多組） | Kruskal-Wallis + post-hoc | `scipy.stats.kruskal` |
| 再入院率（比例, 多院） | 漏斗圖 + z-test | 自製函數 |
| 診斷準確度 | 混淆矩陣, McNemar, ROC | `scipy.stats`, 自製函數 |
| 評估者信度 | Cohen's Kappa | 自製函數 |

---

## 綜合練習：完整分析報告

### 情境
你是某醫療中心的數據分析師，收到以下三項任務：

### 基礎題
1. 某批血壓計出廠前抽檢 30 台，量測值為 `np.random.normal(120.5, 1.8, 30)`。規格要求平均值 = 120 mmHg。請進行品質檢驗。

### 進階題
2. 三個科別（內科、外科、婦產科）各收集 50 份病患出院滿意度問卷（1-5分）。請比較三科的滿意度是否有差異，並找出最佳和最差的科別。

### 挑戰題
3. 設計一個完整的診斷工具評估流程：
   - 模擬 1000 個檢體（盛行率 10%）
   - 新舊兩種試劑的敏感度/特異度各異
   - 繪製 ROC 曲線，計算 AUC
   - 進行 McNemar 檢定比較兩試劑
   - 計算 NNT（需要篩檢多少人才能多找到一個真陽性）

---

## 課程總結

恭喜完成「智慧醫療假設檢定」系列課程！

### 學習歷程回顧

| 課程 | 核心內容 | 關鍵技能 |
|------|----------|----------|
| 01 基礎框架 | 醫療統計思維、效應量、檢定力 | 區分臨床 vs 統計顯著性 |
| 02 臨床試驗 | RCT 分析、Table 1、TOST | 完整臨床試驗數據分析 |
| 03 流行病學 | 風險指標、存活分析、SIR | 流行病學研究方法 |
| 04 產後憂鬱 | 完整案例研究 | 多假設整合分析 |
| 05 醫療品質 | 品管、診斷、信度 | 醫療品質評估 |

### 延伸學習建議
- **進階主題**：多階層模型（Mixed Effects）、Bayesian 方法、Meta-analysis
- **實務應用**：臨床決策支持系統、醫療品質指標儀表板
- **推薦書籍**：
  - Bland,《An Introduction to Medical Statistics》
  - Rosner,《Fundamentals of Biostatistics》